# Rotten Fruits Classification with PyTorch

This notebook provides PyTorch implementations of a simple CNN, a VGG-style CNN, and a VGG-style network with residual blocks.

In [ ]:
# Dependencies
# Run this once in a notebook environment.
# !pip install torch torchvision opendatasets scikit-learn

In [ ]:
import copy
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)

In [ ]:
from pathlib import Path

# Locate the downloaded dataset relative to the repository or current directory.
candidates = [
    Path.cwd() / 'rotten-fruits' / 'dataset',
    Path.cwd() / 'notebooks' / 'rotten-fruits' / 'dataset',
]
for candidate in candidates:
    if candidate.is_dir():
        dataset_dir = candidate
        break
else:
    dataset_dir = candidates[0]

train_dir = dataset_dir / 'train'
test_dir = dataset_dir / 'test'
print('Dataset directory:', dataset_dir)
print('Train exists:', train_dir.is_dir())
print('Test exists:', test_dir.is_dir())

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_data = datasets.ImageFolder(str(train_dir), transform=train_transform)
raw_test_data = datasets.ImageFolder(str(test_dir), transform=eval_transform)

class_names = train_data.classes
class_to_idx = train_data.class_to_idx
missing_from_test = sorted(set(class_names) - set(raw_test_data.classes))
print('Training classes:', class_names)
print('Test classes:', raw_test_data.classes)
print('Classes missing from test:', missing_from_test)

class RemappedSubset(Dataset):
    def __init__(self, imagefolder, target_class_to_idx):
        self.imagefolder = imagefolder
        self.target_class_to_idx = target_class_to_idx

    def __len__(self):
        return len(self.imagefolder.samples)

    def __getitem__(self, index):
        image, source_idx = self.imagefolder[index]
        class_name = self.imagefolder.classes[source_idx]
        target_idx = self.target_class_to_idx[class_name]
        return image, target_idx

# Evaluate only the classes shared by train and test. The original test folder
# has 8 of the 12 training classes, so this prevents label remapping errors.
common_test = RemappedSubset(raw_test_data, class_to_idx)
num_classes = len(class_names)

batch_size = 32
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(common_test, batch_size=batch_size, shuffle=False, num_workers=0)

print('Training samples:', len(train_data))
print('Test samples:', len(common_test))
print('Number of classes:', num_classes)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 54 * 54, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

simple_model = SimpleCNN(num_classes).to(device)
print(simple_model)

In [ ]:
class VGGStyleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

vgg_model = VGGStyleCNN(num_classes).to(device)
print(vgg_model)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class ResidualVGG(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            ResidualBlock(64), ResidualBlock(64),
            nn.Conv2d(64, 128, kernel_size=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block3 = nn.Sequential(
            ResidualBlock(128), ResidualBlock(128), ResidualBlock(128), ResidualBlock(128),
            nn.Conv2d(128, 256, kernel_size=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block4 = nn.Sequential(
            ResidualBlock(256), ResidualBlock(256), ResidualBlock(256), ResidualBlock(256),
            nn.Conv2d(256, 512, kernel_size=1),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 14 * 14, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.classifier(x)

residual_model = ResidualVGG(num_classes).to(device)
print(residual_model)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, epoch, epochs):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    print(f'Epoch {epoch}/{epochs} - loss: {total_loss / total:.4f} - accuracy: {correct / total:.4f}')


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    criterion = nn.CrossEntropyLoss()

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    metrics = {'loss': total_loss / total, 'accuracy': correct / total}
    print(f"Test loss: {metrics['loss']:.4f} - Test accuracy: {metrics['accuracy']:.4f}")
    return metrics


def fit(model, train_loader, test_loader, epochs=10, learning_rate=1e-4):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {'train_loss': [], 'train_accuracy': [], 'test_loss': [], 'test_accuracy': []}

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        model.eval()
        test_loss = 0.0
        test_correct = 0
        test_total = 0
        with torch.inference_mode():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * labels.size(0)
                test_correct += (outputs.argmax(dim=1) == labels).sum().item()
                test_total += labels.size(0)

        train_loss = total_loss / total
        train_acc = correct / total
        val_loss = test_loss / test_total
        val_acc = test_correct / test_total
        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_acc)
        history['test_loss'].append(val_loss)
        history['test_accuracy'].append(val_acc)
        print(f'Epoch {epoch}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - test_loss: {val_loss:.4f} - test_accuracy: {val_acc:.4f}')

    return history

## Train Simple CNN

In [ ]:
simple_history = fit(simple_model, train_loader, test_loader, epochs=10, learning_rate=1e-4)

## Train VGG-style CNN

In [ ]:
vgg_history = fit(vgg_model, train_loader, test_loader, epochs=10, learning_rate=1e-4)

## Train Residual VGG-style CNN

In [ ]:
residual_history = fit(residual_model, train_loader, test_loader, epochs=10, learning_rate=1e-4)